<a href="https://colab.research.google.com/github/LulutheBA/ANLOK-WATER-PREDICTION-PROJECT-/blob/main/NB1_OP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 INSTALL REQUIRED PACKAGES

In [1]:
!pip install pandas numpy matplotlib plotly scikit-learn scipy xgboost shap >/dev/null

print("Required packages installed successfully.")

Required packages installed successfully.


MOUNT GOOGLE DRIVE

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


GITHUB AUTHENTICATION

In [3]:
from getpass import getpass

token = getpass(
    "Enter your GitHub Personal Access Token: "
)

github_user = "LulutheBA"

github_repo = "ANLOK-WATER-PREDICTION-PROJECT-"

print("GitHub credentials received securely.")

Enter your GitHub Personal Access Token: ··········
GitHub credentials received securely.


Clone Repository

In [4]:
import os

repo_path = f"/content/{github_repo}"

# Remove an existing copy if one exists
if os.path.exists(repo_path):
    !rm -rf "$repo_path"

# Clone repository
!git clone https://{token}@github.com/{github_user}/{github_repo}.git

print()
print("Repository cloned successfully.")
print(f"Repository location: {repo_path}")

Cloning into 'ANLOK-WATER-PREDICTION-PROJECT-'...
remote: Enumerating objects: 394, done.
remote: Counting objects: 100% (92/92), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 394 (delta 30), reused 2 (delta 2), pack-reused 302 (from 2)
Receiving objects: 100% (394/394), 6.22 MiB | 6.69 MiB/s, done.
Resolving deltas: 100% (178/178), done.

Repository cloned successfully.
Repository location: /content/ANLOK-WATER-PREDICTION-PROJECT-


VERIFY REPOSITORY

In [5]:
print("=" * 80)
print("REPOSITORY CONTENTS")
print("=" * 80)

if os.path.exists(repo_path):

    for item in os.listdir(repo_path):
        print(item)

else:

    print("Repository was not found.")

REPOSITORY CONTENTS
NB2_OP.ipynb
Notebooks
Copy_of_notebook_1&2.ipynb
README.md
.git
Training
Data
notebook_1.ipynb
Outputs


IMPORT LIBRARIES

In [6]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from scipy.stats import zscore

import glob
import os
import warnings

from IPython.display import display

warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (10, 6)

print("Libraries imported successfully.")

Libraries imported successfully.


LOCATE ALL CSV FILES

In [7]:
csv_files = glob.glob(
    os.path.join(
        repo_path,
        "**",
        "*.csv"
    ),
    recursive=True
)

print(
    f"Found {len(csv_files)} CSV files."
)

print()

for file in csv_files:
    print(file)

Found 28 CSV files.

/content/ANLOK-WATER-PREDICTION-PROJECT-/Data/Raw/NIWIS_Water Supply Reliability - population_16-Jul-2026 .csv
/content/ANLOK-WATER-PREDICTION-PROJECT-/Data/Raw/Water demand growth_ 2026_04_21.csv
/content/ANLOK-WATER-PREDICTION-PROJECT-/Data/Raw/NIWIS_Access to Water Infrastructure Delivered- Population_16-Jul-2026 .csv
/content/ANLOK-WATER-PREDICTION-PROJECT-/Data/Raw/Institutional Compliance_ 2026_04_21.csv
/content/ANLOK-WATER-PREDICTION-PROJECT-/Data/Raw/Microbiological _ Acute Health_ 2026_04_21.csv
/content/ANLOK-WATER-PREDICTION-PROJECT-/Data/Raw/Drinking Water Quality Compliance - SANS 241 _2006 - National [From_ 2022-07-01 To_ 2026-06-30].csv
/content/ANLOK-WATER-PREDICTION-PROJECT-/Data/Raw/Surface Water Storage ( 2020 -2026).csv
/content/ANLOK-WATER-PREDICTION-PROJECT-/Data/Raw/Households Served with Water_ 2026_04_21.csv
/content/ANLOK-WATER-PREDICTION-PROJECT-/Data/Raw/NIWIS_GroundwaterStatus_16-Jul-2026 .csv
/content/ANLOK-WATER-PREDICTION-PROJECT-/D

LOAD DATASETS

In [8]:
datasets = {}

for file in csv_files:

    name = os.path.splitext(
        os.path.basename(file)
    )[0]

    try:

        datasets[name] = pd.read_csv(file)

    except Exception as e:

        print(
            f"Skipping {name}: {e}"
        )

print(
    f"Successfully loaded "
    f"{len(datasets)} datasets."
)

print()

for name, df in datasets.items():

    print(
        f"{name}: {df.shape}"
    )

Skipping Weather4355517: No columns to parse from file
Successfully loaded 27 datasets.

NIWIS_Water Supply Reliability - population_16-Jul-2026 : (144, 14)
Water demand growth_ 2026_04_21: (11, 10)
NIWIS_Access to Water Infrastructure Delivered- Population_16-Jul-2026 : (0, 4)
Institutional Compliance_ 2026_04_21: (11, 5)
Microbiological _ Acute Health_ 2026_04_21: (11, 8)
Drinking Water Quality Compliance - SANS 241 _2006 - National [From_ 2022-07-01 To_ 2026-06-30]: (4, 2)
Surface Water Storage ( 2020 -2026): (375, 9)
Households Served with Water_ 2026_04_21: (11, 17)
NIWIS_GroundwaterStatus_16-Jul-2026 : (1947, 8)
Surface Water Storage (2014-2020): (374, 9)
People Served with Water_ 2026_04_21: (11, 17)
Areas of Highest Vulnerability_ 2026_04_21: (11, 21)
Protests Identified by Municipal IQ_ 2026_04_21: (11, 16)
Verified - River Flow ( National ): (7, 2)
Weather: (21301, 14)
Total Households_ 2026_04_21: (11, 8)
Potable Systems_ 2026_04_21: (61, 11)
Source of Water Households _ 202

DATASET OVERVIEW

In [9]:
overview = []

for name, df in datasets.items():

    overview.append({

        "Dataset": name,

        "Rows": df.shape[0],

        "Columns": df.shape[1],

        "Memory (MB)": round(
            df.memory_usage(
                deep=True
            ).sum() / 1024**2,
            2
        ),

        "Missing Values": int(
            df.isnull().sum().sum()
        ),

        "Duplicate Rows": int(
            df.duplicated().sum()
        ),

        "Numeric Columns": len(
            df.select_dtypes(
                include=np.number
            ).columns
        ),

        "Categorical Columns": len(
            df.select_dtypes(
                exclude=np.number
            ).columns
        )
    })

overview_df = pd.DataFrame(
    overview
)

display(overview_df)

,Dataset,Rows,Columns,Memory (MB),Missing Values,Duplicate Rows,Numeric Columns,Categorical Columns
0,NIWIS_Water Supply Reliability - population_16...,144,14,0.10,189,0,1,13
1,Water demand growth_ 2026_04_21,11,10,0.01,18,0,3,7
2,NIWIS_Access to Water Infrastructure Delivered...,0,4,0.00,0,0,0,4
3,Institutional Compliance_ 2026_04_21,11,5,0.00,13,0,2,3
4,Microbiological _ Acute Health_ 2026_04_21,11,8,0.00,16,0,2,6
5,Drinking Water Quality Compliance - SANS 241 _...,4,2,0.00,0,0,1,1
6,Surface Water Storage ( 2020 -2026),375,9,0.05,1953,0,8,1
7,Households Served with Water_ 2026_04_21,11,17,0.01,25,0,2,15
8,NIWIS_GroundwaterStatus_16-Jul-2026,1947,8,0.69,2511,1,1,7
9,Surface Water Storage (2014-2020),374,9,0.05,1971,0,8,1


INSPECT EVERY DATASET

In [10]:
for name, df in datasets.items():

    print()
    print("=" * 100)
    print(f"DATASET: {name}")
    print("=" * 100)

    print(
        f"Rows: {df.shape[0]}"
    )

    print(
        f"Columns: {df.shape[1]}"
    )

    print("\nColumn names:")

    print(
        list(df.columns)
    )

    print("\nData types:")

    display(
        df.dtypes.to_frame(
            "Data Type"
        )
    )

    print("\nMissing values:")

    missing = df.isnull().sum()

    missing = missing[
        missing > 0
    ]

    if missing.empty:

        print(
            "No missing values."
        )

    else:

        display(
            missing.to_frame(
                "Missing Values"
            )
        )

    print("\nDuplicate rows:")

    print(
        df.duplicated().sum()
    )

    print("\nFirst 5 rows:")

    display(
        df.head()
    )

    print("\nNumerical summary:")

    numeric_cols = df.select_dtypes(
        include=np.number
    ).columns

    if len(numeric_cols) > 0:

        display(
            df[numeric_cols]
            .describe()
            .T
        )

    else:

        print(
            "No numerical columns."
        )


DATASET: NIWIS_Water Supply Reliability - population_16-Jul-2026 
Rows: 144
Columns: 14

Column names:
['Unnamed: 0', 'WSA Name', 'population', 'population with access  to water', 'population Reliable Supply', '% population Reliable Supply', 'Urban population', 'Urban population with access to water', 'Urban population Reliable Supply', '% Urban population Reliable Supply', 'Rural population', 'Rural population  with access  to water', 'Rural population Reliable Supply', '% Rural population Reliable Supply']

Data types:


,Data Type
Unnamed: 0,float64
WSA Name,object
population,object
population with access to water,object
population Reliable Supply,object
% population Reliable Supply,object
Urban population,object
Urban population with access to water,object
Urban population Reliable Supply,object
% Urban population Reliable Supply,object



Missing values:


,Missing Values
Unnamed: 0,144
population with access to water,9
Urban population,2
Urban population with access to water,2
Urban population Reliable Supply,2
% Urban population Reliable Supply,2
Rural population,7
Rural population with access to water,7
Rural population Reliable Supply,7
% Rural population Reliable Supply,7



Duplicate rows:
0

First 5 rows:


...


Missing values:


,Missing Values
Time Frame,1
Unnamed: 2,10
Tariff 0-6kl (incl#VAT),1
Tariff 6-20kl (incl#VAT),1
Tariff 20-60kl (incl#VAT),1
Tariff >60kl (incl#VAT),1



Duplicate rows:
0

First 5 rows:


,Region,Time Frame,Unnamed: 2,Tariff 0-6kl (incl#VAT),Tariff 6-20kl (incl#VAT),Tariff 20-60kl (incl#VAT),Tariff >60kl (incl#VAT)
0,City of Johannesburg Metropolitan Municipality,2 018,NaN,"R0,00","R13,90","R33,73","R38,13"
1,City of Tshwane Metropolitan Municipality,2 018,NaN,"R10,95","R18,89","R29,39","R33,63"
2,Ekurhuleni Metropolitan Municipality,2 018,NaN,"R0,00","R18,30","R26,80","R31,83"
3,Emfuleni,2 018,NaN,"R15,99","R26,94","R39,35","R43,57"
4,Lesedi,2 018,NaN,"R16,06","R19,50","R28,06","R42,45"



Numerical summary:
No numerical columns.


COLUMN-LEVEL AUDIT

In [11]:
column_audit = []

for name, df in datasets.items():

    for col in df.columns:

        column_audit.append({

            "Dataset": name,

            "Column": col,

            "Data Type": str(
                df[col].dtype
            ),

            "Rows": len(df),

            "Missing": int(
                df[col].isnull().sum()
            ),

            "Missing %": round(
                df[col].isnull().mean() * 100,
                2
            ),

            "Unique Values": int(
                df[col].nunique(
                    dropna=True
                )
            ),

            "Duplicate Values": int(
                df[col].duplicated().sum()
            )
        })

column_audit_df = pd.DataFrame(
    column_audit
)

display(
    column_audit_df
)

,Dataset,Column,Data Type,Rows,Missing,Missing %,Unique Values,Duplicate Values
0,NIWIS_Water Supply Reliability - population_16...,Unnamed: 0,float64,144,144,100.00,0,143
1,NIWIS_Water Supply Reliability - population_16...,WSA Name,object,144,0,0.00,144,0
2,NIWIS_Water Supply Reliability - population_16...,population,object,144,0,0.00,144,0
3,NIWIS_Water Supply Reliability - population_16...,population with access to water,object,144,9,6.25,135,8
4,NIWIS_Water Supply Reliability - population_16...,population Reliable Supply,object,144,0,0.00,144,0
...,...,...,...,...,...,...,...,...
308,Residential Tariffs_ 2026_04_21,Unnamed: 2,object,11,10,90.91,1,9
309,Residential Tariffs_ 2026_04_21,Tariff 0-6kl (incl#VAT),object,11,1,9.09,8,2
310,Residential Tariffs_ 2026_04_21,Tariff 6-20kl (incl#VAT),object,11,1,9.09,10,0
311,Residential Tariffs_ 2026_04_21,Tariff 20-60kl (incl#VAT),object,11,1,9.09,10,0


MISSING VALUE AUDIT

In [12]:
missing_columns = column_audit_df[
    column_audit_df["Missing"] > 0
].copy()

print(
    "Columns containing missing values:"
)

display(
    missing_columns
)

Columns containing missing values:


,Dataset,Column,Data Type,Rows,Missing,Missing %,Unique Values,Duplicate Values
0,NIWIS_Water Supply Reliability - population_16...,Unnamed: 0,float64,144,144,100.00,0,143
3,NIWIS_Water Supply Reliability - population_16...,population with access to water,object,144,9,6.25,135,8
6,NIWIS_Water Supply Reliability - population_16...,Urban population,object,144,2,1.39,142,1
7,NIWIS_Water Supply Reliability - population_16...,Urban population with access to water,object,144,2,1.39,142,1
8,NIWIS_Water Supply Reliability - population_16...,Urban population Reliable Supply,object,144,2,1.39,142,1
...,...,...,...,...,...,...,...,...
308,Residential Tariffs_ 2026_04_21,Unnamed: 2,object,11,10,90.91,1,9
309,Residential Tariffs_ 2026_04_21,Tariff 0-6kl (incl#VAT),object,11,1,9.09,8,2
310,Residential Tariffs_ 2026_04_21,Tariff 6-20kl (incl#VAT),object,11,1,9.09,10,0
311,Residential Tariffs_ 2026_04_21,Tariff 20-60kl (incl#VAT),object,11,1,9.09,10,0


DUPLICATE AUDIT

In [13]:
duplicate_summary = (
    overview_df[
        [
            "Dataset",
            "Rows",
            "Duplicate Rows"
        ]
    ]
    .sort_values(
        "Duplicate Rows",
        ascending=False
    )
)

display(
    duplicate_summary
)

,Dataset,Rows,Duplicate Rows
8,NIWIS_GroundwaterStatus_16-Jul-2026,1947,1
1,Water demand growth_ 2026_04_21,11,0
0,NIWIS_Water Supply Reliability - population_16...,144,0
2,NIWIS_Access to Water Infrastructure Delivered...,0,0
3,Institutional Compliance_ 2026_04_21,11,0
5,Drinking Water Quality Compliance - SANS 241 _...,4,0
4,Microbiological _ Acute Health_ 2026_04_21,11,0
6,Surface Water Storage ( 2020 -2026),375,0
7,Households Served with Water_ 2026_04_21,11,0
9,Surface Water Storage (2014-2020),374,0


IDENTIFY DATE/TIME VARIABLES

In [14]:
date_columns = []

for name, df in datasets.items():

    for col in df.columns:

        col_lower = str(
            col
        ).lower()

        if (
            "date" in col_lower
            or "time" in col_lower
            or "timestamp" in col_lower
            or "month" in col_lower
            or "year" in col_lower
        ):

            date_columns.append({

                "Dataset": name,

                "Column": col,

                "Data Type": str(
                    df[col].dtype
                ),

                "Example Values":
                    df[col]
                    .dropna()
                    .astype(str)
                    .head(5)
                    .tolist()
            })

date_columns_df = pd.DataFrame(
    date_columns
)

display(
    date_columns_df
)

,Dataset,Column,Data Type,Example Values
0,Water demand growth_ 2026_04_21,Time Frame,object,"[2 018, 2 018, 2 018, 2 018, 2 018]"
1,Institutional Compliance_ 2026_04_21,Time Frame,object,"[April 2022, April 2022, April 2022, April 202..."
2,Microbiological _ Acute Health_ 2026_04_21,Time Frame,object,"[2 026, 2 026, 2 026, 2 026, 2 026]"
3,Surface Water Storage ( 2020 -2026),'DateTime,object,"['2020-10-01 00:00:00, '2020-10-08 00:00:00, '..."
4,Households Served with Water_ 2026_04_21,Time Frame,object,"[April 2026, April 2026, April 2026, April 202..."
5,Surface Water Storage (2014-2020),'DateTime,object,"['2014-10-01 00:00:00, '2014-10-08 00:00:00, '..."
6,People Served with Water_ 2026_04_21,Time Frame,object,"[April 2026, April 2026, April 2026, April 202..."
7,Areas of Highest Vulnerability_ 2026_04_21,Time Frame,object,"[2 018, 2 018, 2 018, 2 018, 2 018]"
8,Protests Identified by Municipal IQ_ 2026_04_21,Time Frame,object,"[2 018, 2 018, 2 018, 2 018, 2 018]"
9,Weather,DATE,object,"[2000-01-01, 2000-01-02, 2000-01-03, 2000-01-0..."


IDENTIFY LOCATION VARIABLES

In [15]:
location_keywords = [
    "region",
    "municipality",
    "province",
    "district",
    "city",
    "location",
    "area"
]

location_columns = []

for name, df in datasets.items():

    for col in df.columns:

        col_lower = str(
            col
        ).lower()

        if any(
            keyword in col_lower
            for keyword in location_keywords
        ):

            location_columns.append({

                "Dataset": name,

                "Column": col,

                "Unique Values": int(
                    df[col].nunique(
                        dropna=True
                    )
                ),

                "Example Values":
                    df[col]
                    .dropna()
                    .astype(str)
                    .unique()[:5]
                    .tolist()
            })

location_columns_df = pd.DataFrame(
    location_columns
)

display(
    location_columns_df
)

,Dataset,Column,Unique Values,Example Values
0,Water demand growth_ 2026_04_21,Region,11,[City of Johannesburg Metropolitan Municipalit...
1,Institutional Compliance_ 2026_04_21,Region,11,[City of Johannesburg Metropolitan Municipalit...
2,Microbiological _ Acute Health_ 2026_04_21,Region,11,[City of Johannesburg Metropolitan Municipalit...
3,Households Served with Water_ 2026_04_21,Region,11,[City of Johannesburg Metropolitan Municipalit...
4,People Served with Water_ 2026_04_21,Region,11,[City of Johannesburg Metropolitan Municipalit...
5,Areas of Highest Vulnerability_ 2026_04_21,Region,11,[City of Johannesburg Metropolitan Municipalit...
6,Areas of Highest Vulnerability_ 2026_04_21,4# Technical Staff Capacity (Numbers),4,"[Low, Extreme, Moderate, High]"
7,Protests Identified by Municipal IQ_ 2026_04_21,Region,11,[City of Johannesburg Metropolitan Municipalit...
8,Protests Identified by Municipal IQ_ 2026_04_21,Electricity,8,"[76.0, 29.0, 31.0, 1.0, 2.0]"
9,Total Households_ 2026_04_21,Region,11,[City of Johannesburg Metropolitan Municipalit...


NUMERICAL VARIABLE AUDIT

In [16]:
numeric_summary = []

for name, df in datasets.items():

    numeric_cols = df.select_dtypes(
        include=np.number
    ).columns

    for col in numeric_cols:

        numeric_summary.append({

            "Dataset": name,

            "Column": col,

            "Minimum": df[col].min(),

            "Maximum": df[col].max(),

            "Mean": df[col].mean(),

            "Median": df[col].median(),

            "Std": df[col].std(),

            "Missing": int(
                df[col].isnull().sum()
            )
        })

numeric_summary_df = pd.DataFrame(
    numeric_summary
)

display(
    numeric_summary_df
)

,Dataset,Column,Minimum,Maximum,Mean,Median,Std,Missing
0,NIWIS_Water Supply Reliability - population_16...,Unnamed: 0,NaN,NaN,NaN,NaN,NaN,144
1,Water demand growth_ 2026_04_21,% Water Demand Growth,-10.13,8.90,0.651000,0.575,4.909982,1
2,Water demand growth_ 2026_04_21,% Non-Revenue Water,23.00,48.80,33.889000,35.020,7.591215,1
3,Water demand growth_ 2026_04_21,% Water Losses,23.00,48.80,31.864000,30.250,7.504036,1
4,Institutional Compliance_ 2026_04_21,Potable Water Compliance Percentage,62.20,103.51,93.051000,99.540,14.545463,1
...,...,...,...,...,...,...,...,...
65,Operational_ 2026_04_21,Compliance Percentage,0.00,99.19,76.927000,96.990,40.711937,1
66,NIWIS_RWT_16-Jul-2026,Date,2025.00,2025.00,2025.000000,2025.000,0.000000,0
67,NIWIS_RWT_16-Jul-2026,Domestic and Industry c/m3,1.98,6.59,4.581111,4.780,1.488367,0
68,NIWIS_RWT_16-Jul-2026,Irrigation c/m3,1.15,3.84,2.981111,3.220,0.827880,0


OUTLIER AUDIT USING IQR

In [17]:
outlier_summary = []

for name, df in datasets.items():

    numeric_cols = df.select_dtypes(
        include=np.number
    ).columns

    for col in numeric_cols:

        series = df[col].dropna()

        if len(series) < 4:
            continue

        Q1 = series.quantile(0.25)

        Q3 = series.quantile(0.75)

        IQR = Q3 - Q1

        lower_bound = (
            Q1 - 1.5 * IQR
        )

        upper_bound = (
            Q3 + 1.5 * IQR
        )

        outliers = (
            (series < lower_bound)
            |
            (series > upper_bound)
        )

        outlier_summary.append({

            "Dataset": name,

            "Column": col,

            "Q1": Q1,

            "Q3": Q3,

            "IQR": IQR,

            "Lower Bound":
                lower_bound,

            "Upper Bound":
                upper_bound,

            "Outlier Count":
                int(outliers.sum()),

            "Outlier %": round(
                outliers.mean() * 100,
                2
            )
        })

outlier_df = pd.DataFrame(
    outlier_summary
)

display(
    outlier_df.sort_values(
        "Outlier Count",
        ascending=False
    )
)

,Dataset,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count,Outlier %
45,Weather,PRCP,0.000000,4.300000,4.300000,-6.450000,10.750000,1028,12.82
47,Weather,TMAX,22.000000,28.700000,6.700000,11.950000,38.750000,49,0.25
46,Weather,TAVG,14.300000,20.900000,6.600000,4.400000,30.800000,40,0.19
15,Surface Water Storage ( 2020 -2026),'2025/2026,84.365152,94.346891,9.981739,69.392543,109.319499,14,4.64
50,Potable Systems_ 2026_04_21,Drinking Water Quality Percentage,98.390000,100.000000,1.610000,95.975000,102.415000,8,13.33
...,...,...,...,...,...,...,...,...,...
56,Vulnerability Score Per leg_ 2026_04_21,7# Drinking Water Safety & Regulatory Compliance,65.000000,93.500000,28.500000,22.250000,136.250000,0,0.00
60,Piped Water Households _ 2026_04_21,Unspecified,0.000000,0.000000,0.000000,0.000000,0.000000,0,0.00
61,Piped Water Households _ 2026_04_21,Not applicable,0.000000,0.000000,0.000000,0.000000,0.000000,0,0.00
65,NIWIS_RWT_16-Jul-2026,Domestic and Industry c/m3,3.560000,5.480000,1.920000,0.680000,8.360000,0,0.00


MASTER AUDIT REPORT

In [18]:
audit_report = overview_df.copy()

audit_report["Missing %"] = (
    audit_report["Missing Values"]
    / audit_report["Rows"]
    * 100
).round(2)

audit_report["Duplicate %"] = (
    audit_report["Duplicate Rows"]
    / audit_report["Rows"]
    * 100
).round(2)

display(
    audit_report
)

,Dataset,Rows,Columns,Memory (MB),Missing Values,Duplicate Rows,Numeric Columns,Categorical Columns,Missing %,Duplicate %
0,NIWIS_Water Supply Reliability - population_16...,144,14,0.10,189,0,1,13,131.25,0.00
1,Water demand growth_ 2026_04_21,11,10,0.01,18,0,3,7,163.64,0.00
2,NIWIS_Access to Water Infrastructure Delivered...,0,4,0.00,0,0,0,4,NaN,NaN
3,Institutional Compliance_ 2026_04_21,11,5,0.00,13,0,2,3,118.18,0.00
4,Microbiological _ Acute Health_ 2026_04_21,11,8,0.00,16,0,2,6,145.45,0.00
5,Drinking Water Quality Compliance - SANS 241 _...,4,2,0.00,0,0,1,1,0.00,0.00
6,Surface Water Storage ( 2020 -2026),375,9,0.05,1953,0,8,1,520.80,0.00
7,Households Served with Water_ 2026_04_21,11,17,0.01,25,0,2,15,227.27,0.00
8,NIWIS_GroundwaterStatus_16-Jul-2026,1947,8,0.69,2511,1,1,7,128.97,0.05
9,Surface Water Storage (2014-2020),374,9,0.05,1971,0,8,1,527.01,0.00


SAVE AUDIT RESULTS

In [19]:
audit_dir = "Outputs/DataAudit"

os.makedirs(
    audit_dir,
    exist_ok=True
)

overview_df.to_csv(
    os.path.join(
        audit_dir,
        "dataset_overview.csv"
    ),
    index=False
)

column_audit_df.to_csv(
    os.path.join(
        audit_dir,
        "column_audit.csv"
    ),
    index=False
)

missing_columns.to_csv(
    os.path.join(
        audit_dir,
        "missing_columns.csv"
    ),
    index=False
)

duplicate_summary.to_csv(
    os.path.join(
        audit_dir,
        "duplicate_summary.csv"
    ),
    index=False
)

numeric_summary_df.to_csv(
    os.path.join(
        audit_dir,
        "numeric_summary.csv"
    ),
    index=False
)

outlier_df.to_csv(
    os.path.join(
        audit_dir,
        "outlier_summary.csv"
    ),
    index=False
)

audit_report.to_csv(
    os.path.join(
        audit_dir,
        "master_audit_report.csv"
    ),
    index=False
)

print(
    "All audit reports saved successfully."
)

print(
    f"Location: {audit_dir}"
)

All audit reports saved successfully.
Location: Outputs/DataAudit
